# STAKE Crypto Experiment 01 — H0-A Only

**Status: PREREGISTERED / NOT YET RUN**

This notebook is deliberately standalone. Do not import STAKE research-core modules, adapters, economic models, or the sports movement taxonomy into this experiment. The purpose is to answer one question before architecture is built.

> **H0-A:** Cross-venue temporal dislocations in BTC/USDT are not statistically distinguishable from an appropriate null model.

The experiment does **not** test fees, slippage, latency, execution, ROI, CLV, or trading profitability. Those belong to H0-B and remain out of scope.


## 1. Frozen scope

- Asset: **BTC/USDT spot**.
- Venues: **Binance Spot** and **OKX Spot**. If either venue cannot supply a dataset with adequate timestamp provenance, the experiment stops; do not substitute a venue after inspecting results.
- Observation period: **one continuous UTC week**. Freeze the exact start/end timestamps before analysis.
- Data: prefer best-bid/best-ask order-book updates. Trade data may be used only if both venues cannot provide comparable book data, and the substitution must be recorded before analysis.
- No execution simulation.
- No outcome prediction.
- No model fitting to maximize the event count.
- No threshold tuning after seeing the result.


## 2. Data contract and timestamp audit

Every raw record must preserve, where available:

- source/event timestamp;
- exchange-provided receive timestamp, if supplied;
- local ingestion timestamp, if supplied;
- sequence/update identifier;
- venue;
- symbol;
- bid/ask or trade price;
- bid/ask size when available.

The notebook must report the provenance of every clock. A local collection timestamp must never be relabeled as exchange-event time.

**Gate 0:** if the timestamp resolution or provenance cannot support the intended temporal comparison, stop the experiment and report the data limitation. Do not proceed on interpolated or invented timing.


In [ ]:
# Environment only. Run before loading market data.
from pathlib import Path
import platform, sys
import numpy as np
import pandas as pd

print('Python:', sys.version)
print('Platform:', platform.platform())
print('pandas:', pd.__version__)

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data' / 'crypto_01'
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_DIR:', DATA_DIR.resolve())

## 3. Raw-data acquisition

Acquire exactly one continuous week from both venues. Prefer official public market-data endpoints/files. Store the untouched raw files under `data/crypto_01/raw/`. Record the acquisition URL/source, download time, file checksum, and exact UTC coverage.

The acquisition step must not silently backfill missing records with a different source. If a week cannot be reconstructed faithfully, stop.

The notebook intentionally does not contain a live trading client. It is a measurement experiment, not an execution system.

In [ ]:
# Expected normalized input columns. Adapt ONLY the raw-loader cells to the downloaded files.
EXPECTED = [
    'venue', 'symbol', 'source_time_ns', 'ingest_time_ns',
    'sequence_id', 'bid_px', 'ask_px', 'bid_sz', 'ask_sz'
]

def assert_schema(df):
    missing = [c for c in EXPECTED if c not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    if df.empty:
        raise ValueError('Empty dataset')
    if (df['ask_px'] < df['bid_px']).any():
        raise ValueError('Crossed/invalid top-of-book records detected')

# Example after writing the two raw loaders:
# binance = load_binance_raw(...)
# okx = load_okx_raw(...)
# df = pd.concat([binance, okx], ignore_index=True)
# assert_schema(df)

## 4. Gate 0 — timestamp audit

Run this before calculating any dislocation statistic.

Required outputs:

1. exact UTC start/end coverage for each venue;
2. source timestamp unit and resolution;
3. duplicate timestamp rate;
4. sequence gaps / out-of-order updates;
5. median, p95 and minimum inter-update gap per venue;
6. same-timestamp collision rate across venues;
7. missing-data intervals;
8. fraction of observations for which both venues can be aligned;
9. whether 100 ms, 250 ms, 500 ms, 1 s, 5 s and 30 s horizons are actually measurable.

If the data cannot distinguish the ordering required by the event definition below, the result is **UNMEASURABLE**, not a null finding.

In [ ]:
def timestamp_audit(df):
    out = {}
    for venue, g in df.groupby('venue'):
        t = pd.to_datetime(g['source_time_ns'], unit='ns', utc=True).sort_values()
        gaps_ms = t.diff().dt.total_seconds().dropna() * 1000
        out[venue] = {
            'rows': int(len(g)),
            'start': str(t.min()),
            'end': str(t.max()),
            'duplicate_timestamp_rate': float(g['source_time_ns'].duplicated().mean()),
            'min_gap_ms': float(gaps_ms.min()) if len(gaps_ms) else None,
            'median_gap_ms': float(gaps_ms.median()) if len(gaps_ms) else None,
            'p95_gap_ms': float(gaps_ms.quantile(.95)) if len(gaps_ms) else None,
        }
    return pd.DataFrame(out).T

# audit = timestamp_audit(df)
# display(audit)

## 5. Frozen dislocation definition

A dislocation is defined **before looking at the study-week results** as follows:

1. Construct each venue's top-of-book midprice: `mid = (bid + ask) / 2`.
2. Align both venues onto a fixed **100 ms** grid using the most recent quote at or before each grid timestamp. Do not forward-fill across a gap longer than **500 ms**.
3. Compute the signed cross-venue log-price gap in basis points: `gap_bps = 10,000 * (log(mid_A) - log(mid_B))`.
4. A candidate dislocation begins when `abs(gap_bps) >= 5 bps` after both venues have valid observations within the 500 ms freshness limit.
5. A candidate is retained only if the gap remains at or above **5 bps for at least 200 ms**.
6. The leader is the venue whose midprice moved first in the direction required to create the gap. If the two qualifying moves cannot be ordered at the available timestamp resolution, classify the event as **SIMULTANEOUS/AMBIGUOUS** and exclude it from directional leader/follower analysis.
7. Events are de-duplicated: after an event begins, no new event may begin for **1 second** on the same venue pair until the gap has returned below 5 bps.

These values are frozen for H0-A. They may be changed only in a separately labeled exploratory replication, never retroactively inside the confirmatory result.


### Why this definition is deliberately conservative

The 5 bps threshold is not being presented as an economically tradable threshold. H0-A does not ask whether 5 bps is profitable. It is a fixed event boundary chosen before the data is inspected so that the experiment tests a reproducible cross-venue separation rather than tuning a threshold until events appear.

The null is tested against controls, not against an assumption that every 5 bps gap is meaningful.

## 6. H0-A test

The primary question is whether the observed dislocations exhibit temporal structure beyond an appropriate null.

For every retained event, measure:

- leader venue;
- absolute and signed gap at event start;
- time from leader move to follower convergence;
- time for the gap to fall below 2.5 bps;
- maximum gap after event start;
- reversal/overshoot within 30 seconds;
- fraction of events resolved in 100 ms / 250 ms / 500 ms / 1 s / 5 s / 30 s, subject to Gate 0.

### Primary null controls

1. **Venue-label permutation:** randomly swap venue identities within each event while preserving event times and magnitudes.
2. **Circular time shift:** shift one venue's time series by a fixed random offset larger than the maximum tested lead window, preserving its internal autocorrelation.
3. **Event-time permutation:** randomly permute event timestamps within the valid study interval while preserving the event-size distribution.

The observed statistic is compared with the empirical null distribution. Report effect size and confidence interval, not only a p-value.

## 7. Primary statistics

The confirmatory report will contain:

- number of valid grid points;
- number of candidate dislocations;
- number of retained dislocations;
- simultaneous/ambiguous fraction;
- observed leader share by venue;
- median and distribution of resolution time;
- observed-vs-null difference in resolution probability by horizon;
- observed-vs-null difference in directional convergence;
- bootstrap confidence intervals using event-level resampling;
- robustness to excluding the largest 1% of gaps;
- robustness across the two venue directions.

Do not call a result a pass merely because it is statistically significant. H0-A concerns distinguishable structure. The report must show the magnitude and whether the effect survives the null controls.

## 8. Decision rule — H0-A only

### PASS / structure detected
Declare H0-A rejected only if:

- Gate 0 passes;
- the preregistered dislocation definition yields a non-trivial number of valid events;
- at least one primary temporal statistic is materially different from all required null controls;
- the direction is consistent across both venue directions; and
- the effect is not explained by timestamp collisions, stale quotes, missing-data artifacts, or one extreme tail.

### FAIL / no distinguishable structure
Declare H0-A not rejected if the observed statistics remain within the null distribution after the required controls.

### UNMEASURABLE
If Gate 0 fails, do not label the mechanism absent. The correct conclusion is that this dataset cannot test it.

H0-B is untouched regardless of the outcome. No profitability conclusion may appear in this notebook.

## 9. Negative-result paper template

After the experiment is run, complete this section even if the result is null.

**Question:** Do cross-venue BTC/USDT price dislocations contain reproducible temporal structure beyond null controls?

**Data:** [venue pair, exact UTC week, source, row counts, timestamp provenance]

**Gate 0:** [PASS / UNMEASURABLE + evidence]

**Frozen event definition:** 5 bps cross-venue mid-price gap, 200 ms persistence, 100 ms alignment grid, 500 ms quote freshness.

**Observed:** [event count, resolution distribution, leader split]

**Null controls:** [results]

**H0-A conclusion:** [REJECT / NOT REJECT / UNMEASURABLE]

**What this does not establish:** No statement about profitability, execution, fees, slippage, or H0-B.

**Next action:** [stop / improve measurement / replicate out-of-sample / earn architecture step]

## 10. Explicit freeze

Until this notebook produces an H0-A result, do **not**:

- rewrite `PROJECT_BIBLE.md` around the crypto pivot;
- build generic market adapters;
- build an economic engine;
- build a Binance/OKX trading bot;
- create a prediction-market adapter;
- tune event thresholds for event count;
- use sports results as evidence for crypto;
- describe detected gaps as smart money or informed flow.

**Architecture is earned by evidence.**